### Transformation

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from typing import List
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
loan_data = spark.read.table("pysparkdbt.bronze.loan_data")
repayment_data = spark.read.table("pysparkdbt.bronze.repayment_data")
sch_loan = spark.read.table("pysparkdbt.bronze.sch_loan")

In [0]:

def profile_df(df, name):
    total_count = df.count()
    dup_count = total_count - df.dropDuplicates().count()

    null_counts = df.select([
        sum(
            when(col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in df.columns
    ]).first().asDict()

    print(f"\n{'='*40}")
    print(f"DataFrame Profile: {name}")
    print(f"{'='*40}")
    print(f"Total Rows: {total_count}")
    print(f"Duplicate Rows: {dup_count}")
    print("\nNull Count By Column:")

    for column, null_count in null_counts.items():
        print(f"{column}: {null_count}")

    print("\nSchema:")
    df.printSchema()

In [0]:

class transformation:

    # 1. Remove duplicate records and keep the latest record
    def dup(self,df: DataFrame,dedup_cols: List[str],cdc: str) -> DataFrame:

        # Combine the selected key columns into one deduplication key
        # Example: loan_id=101 and customer_id=10 -> "101-10"
        df = df.withColumn(
            "dup_col",
            concat_ws("-", *[col(c) for c in dedup_cols]))

        df = df.withColumn(
            "dup_count",
            row_number().over(
                Window.partitionBy("dup_col").orderBy(col(cdc).desc()) ))

        df = df.filter(col("dup_count") == 1)
        df = df.drop("dup_count", "dup_col")
        return df

    # 2. Add the processing timestamp to every row
    def process_timestamp(self, df: DataFrame) -> DataFrame:

        # Record when this transformation is processed
        df = df.withColumn(
            "process_timestamp",
            current_timestamp())

        return df

    # 3. Insert new records and update existing records in a Delta table
    def upsert(self,df: DataFrame, key_cols: List[str],table: str, cdc: str):

        # Build the matching condition using business key columns.
        # Example: src.loan_id = trg.loan_id
        merge_condition = " AND ".join(
            [f"src.{i} = trg.{i}" for i in key_cols]
        )

        # Load the existing target Delta table.
        dlt_obj = DeltaTable.forName(spark, f"pysparkdbt.silver.{table}")

        # Compare incoming source records with existing target records
        dlt_obj.alias("trg").merge(df.alias("src"),merge_condition) \
        .whenMatchedUpdateAll( condition=f"src.{cdc} >= trg.{cdc}" ) \
        .whenNotMatchedInsertAll() \
        .execute()

     
        return "Done"


In [0]:
profile_df(loan_data, "loan_data")
profile_df(repayment_data, "repayment_loan")
profile_df(sch_loan, "schedule_loan")


DataFrame Profile: loan_data
Total Rows: 100
Duplicate Rows: 0

Null Count By Column:
loan_id: 0
customer_id: 0
loan_type: 0
loan_amount: 0
interest_rate_pct: 0
tenure_months: 0
disbursement_date: 0
loan_status: 0

Schema:
root
 |-- loan_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- loan_type: string (nullable = true)
 |-- loan_amount: integer (nullable = true)
 |-- interest_rate_pct: double (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- disbursement_date: date (nullable = true)
 |-- loan_status: string (nullable = true)


DataFrame Profile: repayment_loan
Total Rows: 300
Duplicate Rows: 0

Null Count By Column:
repayment_id: 0
loan_id: 0
due_date: 0
payment_date: 120
amount_due: 0
amount_paid: 0
payment_status: 0

Schema:
root
 |-- repayment_id: string (nullable = true)
 |-- loan_id: string (nullable = true)
 |-- due_date: date (nullable = true)
 |-- payment_date: date (nullable = true)
 |-- amount_due: integer (nullable = true)
 |

### Loan Data

In [0]:


loan_data = loan_data.withColumn("od_bucket",
                     when(col("loan_status").isin("Active", "Closed"),"0 Bucket")\
                     .otherwise("0+ Bucket"))
                     
loan_data = loan_data.withColumn(
    "last_update_timestamp",
    current_timestamp()
)




In [0]:
transform = transformation()


loan_data_obj=transform.dup(loan_data,['loan_id'],'last_update_timestamp')
loan_data_obj=transform.process_timestamp(loan_data_obj)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.loan_data"):

    loan_data_obj.write.format("delta") \
        .mode("append") \
        .saveAsTable("pysparkdbt.silver.loan_data")

else:

    transform.upsert(
        loan_data_obj,
        ["loan_id"],
        "loan_data",
        "last_update_timestamp"
    )

### Repayment Data

In [0]:
repayment_data = repayment_data.withColumn(
    "last_update_timestamp",
    current_timestamp()
)

repay_data_obj=transform.dup(repayment_data,['repayment_id'],'last_update_timestamp')
reoay_data_obj=transform.process_timestamp(repay_data_obj)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.repayment_data"):

    repay_data_obj.write.format("delta") \
        .mode("append") \
        .saveAsTable("pysparkdbt.silver.repaymemt_data")

else:

    transform.upsert(
        repay_data_obj,
        ["repayment_id"],
        "repaymnet_data",
        "last_update_timestamp"
    )

### Loan Schedule Detail

In [0]:
sch_loan = sch_loan.withColumn(
    "last_update_timestamp",
    current_timestamp()
)

sch_data_obj=transform.dup(sch_loan,['schedule_id'],'last_update_timestamp')
sch_data_obj=transform.process_timestamp(sch_data_obj)



In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.sch_loan"):

    sch_data_obj.write.format("delta") \
        .mode("append") \
        .saveAsTable("pysparkdbt.silver.sch_loan")

else:

    transform.upsert(
        sch_data_obj,
        ["schedule_id"],
        "sch_loan",
        "last_update_timestamp"
    )